# One-Dimensional Quantile Quantization Rates

This notebook generates `fig:semidiscrete-quantile-quantization-rates`.  In one dimension, an equal-weight empirical approximation can be optimized directly in quantile coordinates.  If
\[
    Q(u)=F_\alpha^{-1}(u),\qquad I_i=((i-1)/m,i/m],
\]
then the best atom with mass \(1/m\) on the interval \(I_i\) is the average of \(Q\) over that interval.  Midpoint inverse-CDF particles \(Q((i-1/2)/m)\) have the same leading asymptotics, while iid samples converge much more slowly on average.

In [1]:
from pathlib import Path
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE, RED, VIOLET, GRAY, LIGHT_GRAY,
    box_axes, figure_dir, interp_color, save_pdf, setup_matplotlib,
)

setup_matplotlib()

NAME = "semidiscrete-quantile-quantization-rates"
OUT = figure_dir(NAME)
THUMB = ROOT / "notebooks-figures" / "thumbnails" / f"{NAME}.png"
THUMB.parent.mkdir(parents=True, exist_ok=True)
ARXIV_OUT = ROOT / "arxiv" / "figures"
ARXIV_OUT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(20260620)

## Smooth compactly supported law

The density is positive on \([0,1]\), so its quantile derivative is bounded.  This keeps the asymptotic constants finite and makes the deterministic \(m^{-2}\) squared-error law visible at modest resolutions.

In [2]:
x = np.linspace(0.0, 1.0, 16001)


def bump(x, mu, sigma, amp):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

rho = (
    0.18
    + bump(x, 0.22, 0.060, 1.20)
    + bump(x, 0.55, 0.105, 0.92)
    + bump(x, 0.80, 0.045, 0.70)
)
rho = rho / np.trapezoid(rho, x)

dx = np.diff(x)
cdf = np.concatenate([[0.0], np.cumsum(0.5 * (rho[:-1] + rho[1:]) * dx)])
cdf = cdf / cdf[-1]
cdf[0] = 0.0
cdf[-1] = 1.0


def Q(u):
    u = np.clip(np.asarray(u), 0.0, 1.0)
    return np.interp(u, cdf, x)


def sample_alpha(n, trials=1):
    u = rng.random((trials, n))
    return Q(u)

# Smooth asymptotic constants, computed only for reference/guide lines.
u_const = np.linspace(1e-5, 1 - 1e-5, 20000)
q_const = Q(u_const)
qprime = np.gradient(q_const, u_const)
constant_deterministic = np.trapezoid(qprime**2, u_const) / 12.0
constant_random = np.trapezoid(u_const * (1 - u_const) * qprime**2, u_const)
print(f"deterministic squared-W2 constant: {constant_deterministic:.4e}")
print(f"iid squared-W2 constant: {constant_random:.4e}")

deterministic squared-W2 constant: 1.2157e-01
iid squared-W2 constant: 1.3427e-01


## Deterministic and iid errors

For each \(m\), deterministic errors are evaluated by quadrature on each quantile bin.  The random curve averages the exact one-dimensional Wasserstein error of many iid empirical measures, computed after sorting the samples.

In [3]:
m_values = np.array([8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384])
quad_per_bin = 64

errors_bin_mean = []
errors_midpoint = []
errors_random = []
errors_random_q25 = []
errors_random_q75 = []
trial_counts = []

for m in m_values:
    local = (np.arange(quad_per_bin) + 0.5) / quad_per_bin
    u_bins = (np.arange(m)[:, None] + local[None, :]) / m
    q_bins = Q(u_bins)

    y_mean = q_bins.mean(axis=1)
    y_mid = Q((np.arange(m) + 0.5) / m)
    errors_bin_mean.append(np.mean((q_bins - y_mean[:, None]) ** 2))
    errors_midpoint.append(np.mean((q_bins - y_mid[:, None]) ** 2))

    trials = int(max(160, min(1000, round(6200 / m))))
    samples = np.sort(sample_alpha(m, trials=trials), axis=1)
    err = np.mean((samples[:, :, None] - q_bins[None, :, :]) ** 2, axis=(1, 2))
    errors_random.append(float(err.mean()))
    errors_random_q25.append(float(np.quantile(err, 0.25)))
    errors_random_q75.append(float(np.quantile(err, 0.75)))
    trial_counts.append(trials)

errors_bin_mean = np.asarray(errors_bin_mean)
errors_midpoint = np.asarray(errors_midpoint)
errors_random = np.asarray(errors_random)
errors_random_q25 = np.asarray(errors_random_q25)
errors_random_q75 = np.asarray(errors_random_q75)
trial_counts = np.asarray(trial_counts)
print("m values:", m_values)
print("iid trials:", trial_counts)

m values: [  8  12  16  24  32  48  64  96 128 192 256 384]
iid trials: [775 517 388 258 194 160 160 160 160 160 160 160]


## Exported panels

The first panel shows the target density and two equal-cardinality point sets: optimized inverse-CDF bin averages and one iid draw.  The second panel displays the mean squared Wasserstein error decay.

In [4]:
def finish_density_axis(ax):
    box_axes(ax)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(-0.58, 1.08 * rho.max())
    ax.set_yticks([])
    ax.set_xlabel(r"$x$")
    ax.tick_params(axis="x", labelsize=8, pad=1)

m_display = 22
local_display = (np.arange(128) + 0.5) / 128
u_display = (np.arange(m_display)[:, None] + local_display[None, :]) / m_display
y_equal = Q(u_display).mean(axis=1)
y_iid = np.sort(sample_alpha(m_display, trials=1)[0])
colors_equal = [interp_color(i / (m_display - 1), RED, BLUE) for i in range(m_display)]

fig, ax = plt.subplots(figsize=(3.05, 1.95))
ax.fill_between(x, 0, rho, color=LIGHT_GRAY, alpha=0.38, linewidth=0)
ax.plot(x, rho, color=GRAY, lw=0.95, alpha=0.85)
ax.scatter(y_equal, -0.16 * np.ones_like(y_equal), s=14, color=colors_equal, edgecolor="white", linewidth=0.28, zorder=4)
ax.scatter(y_iid, -0.36 * np.ones_like(y_iid), s=11, color=GRAY, alpha=0.72, edgecolor="none", zorder=3)
finish_density_axis(ax)
save_pdf(fig, OUT / "quantile-points.pdf", pad_inches=0.025)
plt.close(fig)

fig, ax = plt.subplots(figsize=(3.05, 1.95))
ax.loglog(m_values, errors_random, "o-", color=RED, lw=1.15, ms=3.2, label=r"iid average")
ax.fill_between(m_values, errors_random_q25, errors_random_q75, color=RED, alpha=0.11, linewidth=0)
ax.loglog(m_values, errors_midpoint, "o-", color=VIOLET, lw=1.05, ms=3.0, label=r"midpoint quantiles")
ax.loglog(m_values, errors_bin_mean, "o-", color=BLUE, lw=1.15, ms=3.0, label=r"bin averages")

m_guide = np.array([m_values[1], m_values[-1]])
guide_random = errors_random[1] * (m_guide / m_values[1]) ** (-1)
guide_det = errors_bin_mean[1] * (m_guide / m_values[1]) ** (-2)
ax.loglog(m_guide, guide_random, "--", color=GRAY, lw=0.75, alpha=0.60)
ax.loglog(m_guide, guide_det, "--", color=GRAY, lw=0.75, alpha=0.60)
ax.text(m_guide[-1] * 0.88, guide_random[-1] * 1.18, r"$m^{-1}$", color=GRAY, fontsize=7.2, ha="right")
ax.text(m_guide[-1] * 0.88, guide_det[-1] * 1.45, r"$m^{-2}$", color=GRAY, fontsize=7.2, ha="right")

box_axes(ax)
ax.set_xlabel(r"number of atoms $m$")
ax.set_ylabel(r"squared $W_2$ error")
ax.grid(True, which="major", color="#d7d7d7", lw=0.45, alpha=0.55)
ax.grid(True, which="minor", color="#ececec", lw=0.30, alpha=0.50)
ax.legend(loc="lower left", frameon=False, fontsize=7.1, handlelength=1.4, borderpad=0.1)
ax.tick_params(labelsize=7.5, pad=1.5)
save_pdf(fig, OUT / "rate-comparison.pdf", pad_inches=0.025)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(6.55, 2.05), gridspec_kw={"wspace": 0.26})
ax = axes[0]
ax.fill_between(x, 0, rho, color=LIGHT_GRAY, alpha=0.38, linewidth=0)
ax.plot(x, rho, color=GRAY, lw=0.95, alpha=0.85)
ax.scatter(y_equal, -0.16 * np.ones_like(y_equal), s=14, color=colors_equal, edgecolor="white", linewidth=0.28, zorder=4)
ax.scatter(y_iid, -0.36 * np.ones_like(y_iid), s=11, color=GRAY, alpha=0.72, edgecolor="none", zorder=3)
finish_density_axis(ax)

ax = axes[1]
ax.loglog(m_values, errors_random, "o-", color=RED, lw=1.15, ms=3.2)
ax.fill_between(m_values, errors_random_q25, errors_random_q75, color=RED, alpha=0.11, linewidth=0)
ax.loglog(m_values, errors_midpoint, "o-", color=VIOLET, lw=1.05, ms=3.0)
ax.loglog(m_values, errors_bin_mean, "o-", color=BLUE, lw=1.15, ms=3.0)
ax.loglog(m_guide, guide_random, "--", color=GRAY, lw=0.75, alpha=0.60)
ax.loglog(m_guide, guide_det, "--", color=GRAY, lw=0.75, alpha=0.60)
box_axes(ax)
ax.set_xlabel(r"number of atoms $m$")
ax.set_ylabel(r"squared $W_2$ error")
ax.grid(True, which="major", color="#d7d7d7", lw=0.45, alpha=0.55)
ax.grid(True, which="minor", color="#ececec", lw=0.30, alpha=0.50)
ax.tick_params(labelsize=7.5, pad=1.5)
fig.savefig(THUMB, dpi=180, bbox_inches="tight", pad_inches=0.045)
plt.close(fig)

for pdf in OUT.glob("*.pdf"):
    shutil.copy2(pdf, ARXIV_OUT / f"{NAME}--{pdf.name}")

print(f"wrote {OUT}")
print(f"thumbnail: {THUMB}")

'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


wrote /Users/gpeyre/Dropbox/github/ot4ml/latex/figures/semidiscrete-quantile-quantization-rates
thumbnail: /Users/gpeyre/Dropbox/github/ot4ml/notebooks-figures/thumbnails/semidiscrete-quantile-quantization-rates.png
